# Interpolation Search

Objetivo: Estimar a posicao provavel do alvo em uma colecao **ordenada e com distribuicao uniforme** usando interpolacao linear, em vez de sempre ir ao ponto medio.

Complexidade:
- Tempo: O(log log n) dados uniformemente distribuidos | O(n) pior caso (dados muito distorcidos)
- Espaco: O(1)

Categoria: Busca

## Diagrama do fluxo

```mermaid
flowchart TD
    A[Inicio: low=0, high=n-1] --> B{low <= high AND arr low <= target <= arr high ?}
    B -->|Nao| C[Retornar -1]
    B -->|Sim| D["pos = low + (target - arr[low]) * (high - low) / (arr[high] - arr[low])"]
    D --> E{arr pos == target ?}
    E -->|Sim| F[Retornar pos]
    E -->|Nao| G{arr pos < target ?}
    G -->|Sim| H[low = pos + 1]
    G -->|Nao| I[high = pos - 1]
    H --> B
    I --> B
```

## Fundamentos

Binary search sempre vai ao ponto medio. Mas humanos fazem diferente: quando buscamos a palavra "zoo" em um dicionario, nao abrimos no meio -- vamos direto ao final. Interpolation search imita esse comportamento.

A formula de estimacao e derivada da **interpolacao linear** entre os extremos:

```
pos = low + (target - arr[low]) * (high - low) / (arr[high] - arr[low])
```

Para dados **uniformemente distribuidos**, essa estimativa e muito precisa e o algoritmo converge em O(log log n) iteracoes. Para dados **distorcidos** (ex: valores exponencialmente crescentes), a estimativa erra sistematicamente e a complexidade pode degradar para O(n).

Casos de uso reais incluem busca em tabelas de lookup de metricas uniformes (ex: percentis de latencia), indices de banco de dados com distribuicao conhecida, e pesquisa em tabelas de roteamento com IPs densamente distribuidos.

In [ ]:
def interpolation_search(arr, target):
    """
    Searches for target in a sorted array using interpolation search.
    Best for uniformly distributed data.

    Args:
        arr: Sorted list of numeric elements.
        target: Numeric value to search for.

    Returns:
        Index of target, or -1 if not found.

    Time:  O(log log n) uniform data | O(n) worst case (skewed data)
    Space: O(1)
    """
    low, high = 0, len(arr) - 1

    while (
        low <= high
        and arr[low] <= target <= arr[high]
    ):
        # Guard against division by zero when all elements in range are equal
        if arr[high] == arr[low]:
            return low if arr[low] == target else -1

        # Interpolation formula: estimate position proportionally
        pos = low + int(
            (target - arr[low]) * (high - low) / (arr[high] - arr[low])
        )

        if arr[pos] == target:
            return pos
        elif arr[pos] < target:
            low = pos + 1
        else:
            high = pos - 1

    return -1


# Validation on uniformly distributed data
uniform = list(range(10, 210, 10))  # [10, 20, 30, ..., 200]
print(f"Uniform array (step=10): {uniform}")
print()
print(f"interpolation_search(arr, 80):  {interpolation_search(uniform, 80)}")    # 7
print(f"interpolation_search(arr, 10):  {interpolation_search(uniform, 10)}")    # 0
print(f"interpolation_search(arr, 200): {interpolation_search(uniform, 200)}")   # 19
print(f"interpolation_search(arr, 15):  {interpolation_search(uniform, 15)}")    # -1

## Visualizacao: calculo da posicao estimada

In [ ]:
def interpolation_search_verbose(arr, target):
    """
    Interpolation search with step-by-step trace showing estimated positions.
    Returns (index_or_minus1, iterations).
    """
    low, high = 0, len(arr) - 1
    iteration = 0

    print(f"Searching for {target} | n={len(arr)}")
    print(f"{'Iter':>5} | {'low':>5} | {'high':>5} | {'pos (est)':>10} | {'arr[pos]':>10} | {'action':>20}")
    print("-" * 67)

    while low <= high and arr[low] <= target <= arr[high]:
        iteration += 1

        if arr[high] == arr[low]:
            result = low if arr[low] == target else -1
            print(f"All equal in range, result={result}")
            return result, iteration

        pos = low + int((target - arr[low]) * (high - low) / (arr[high] - arr[low]))

        if arr[pos] == target:
            print(f"{iteration:>5} | {low:>5} | {high:>5} | {pos:>10} | {arr[pos]:>10} | {'FOUND':>20}")
            print(f"\nResult: index {pos} after {iteration} iteration(s).")
            return pos, iteration
        elif arr[pos] < target:
            action = f"low = {pos + 1}"
            print(f"{iteration:>5} | {low:>5} | {high:>5} | {pos:>10} | {arr[pos]:>10} | {action:>20}")
            low = pos + 1
        else:
            action = f"high = {pos - 1}"
            print(f"{iteration:>5} | {low:>5} | {high:>5} | {pos:>10} | {arr[pos]:>10} | {action:>20}")
            high = pos - 1

    print(f"\nResult: -1 (not found or out of range) after {iteration} iteration(s).")
    return -1, iteration


uniform = list(range(0, 1000, 10))  # 100 elements: [0, 10, 20, ..., 990]
interpolation_search_verbose(uniform, 470)

## Quando supera o binary search e quando degrada

A eficiencia do interpolation search depende fortemente da distribuicao dos dados. Abaixo demonstramos os dois extremos com colecoes reais:
- **Distribuicao uniforme**: salarios mensais iguais, timestamps com intervalo fixo
- **Distribuicao distorcida**: acessos a paginas web (lei de Zipf), latencias com outliers extremos

In [ ]:
import random


def count_iterations_interpolation(arr, target):
    """Returns number of iterations for interpolation search."""
    low, high = 0, len(arr) - 1
    iterations = 0
    while low <= high and arr[low] <= target <= arr[high]:
        iterations += 1
        if arr[high] == arr[low]:
            return iterations
        pos = low + int((target - arr[low]) * (high - low) / (arr[high] - arr[low]))
        if arr[pos] == target:
            return iterations
        elif arr[pos] < target:
            low = pos + 1
        else:
            high = pos - 1
    return iterations


def count_iterations_binary(arr, target):
    """Returns number of iterations for binary search."""
    low, high = 0, len(arr) - 1
    iterations = 0
    while low <= high:
        iterations += 1
        mid = low + (high - low) // 2
        if arr[mid] == target:
            return iterations
        elif arr[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return iterations


N = 10_000

# Uniform: integers 0..N spaced evenly
uniform = sorted(random.sample(range(N), N // 2))

# Skewed: exponential distribution (many small values, few very large)
skewed = sorted(int(random.expovariate(1 / 100)) for _ in range(N // 2))
# Remove duplicates to make binary search valid index comparison fair
skewed = sorted(set(skewed))

datasets = [
    ("Uniform",  uniform),
    ("Skewed (exponential)", skewed),
]

print(f"{'Dataset':>25} | {'n':>7} | {'interp iters':>14} | {'binary iters':>14} | {'winner':>10}")
print("-" * 80)

for name, arr in datasets:
    # Pick a target from the second half of the array
    target = arr[len(arr) * 3 // 4]
    iters_i = count_iterations_interpolation(arr, target)
    iters_b = count_iterations_binary(arr, target)
    winner = "interpolation" if iters_i < iters_b else "binary" if iters_b < iters_i else "tie"
    print(f"{name:>25} | {len(arr):>7} | {iters_i:>14} | {iters_b:>14} | {winner:>10}")

## Analise de Complexidade

### Tempo

**O(log log n)** -- dados uniformemente distribuidos. A cada iteracao a faixa de busca e reduzida de forma muito mais agressiva que o binary search (que reduz pela metade). Com dados uniformes, a reducao e aproximadamente proporcional a distribuicao.

**O(n)** -- pior caso. Ocorre quando os dados sao extremamente distorcidos (ex: potencias de 2), fazendo a estimativa apontar para posicoes distantes do alvo em todas as iteracoes.

### Espaco

**O(1)** -- apenas variaveis `low`, `high`, `pos`.

### Comparacao de complexidade

| Algoritmo | Melhor caso | Caso medio | Pior caso |
|---|---|---|---|
| Linear Search | O(1) | O(n) | O(n) |
| Binary Search | O(1) | O(log n) | O(log n) |
| Jump Search | O(1) | O(sqrt n) | O(sqrt n) |
| Interpolation Search | O(1) | O(log log n) uniforme | O(n) distorcido |

**Regra pratica**: use interpolation search apenas quando puder garantir que os dados sao aproximadamente uniformes. Caso contrario, binary search oferece O(log n) garantido sem dependencia da distribuicao.

In [ ]:
import timeit
import random

SIZES = [10_000, 100_000, 1_000_000]
RUNS = 300


def binary_search_for_bench(arr, target):
    low, high = 0, len(arr) - 1
    while low <= high:
        mid = low + (high - low) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return -1


print("=== Distribuicao UNIFORME ===")
print(f"{'n':>12} | {'interpolation (ms)':>20} | {'binary (ms)':>13}")
print("-" * 52)

for n in SIZES:
    uniform = list(range(0, n * 2, 2))  # perfectly uniform, step=2
    target = uniform[n // 2]

    t_interp = timeit.timeit(
        lambda d=uniform, t=target: interpolation_search(d, t), number=RUNS
    ) / RUNS * 1000

    t_binary = timeit.timeit(
        lambda d=uniform, t=target: binary_search_for_bench(d, t), number=RUNS
    ) / RUNS * 1000

    print(f"{n:>12,} | {t_interp:>20.4f} | {t_binary:>13.4f}")

print()
print("=== Distribuicao DISTORCIDA (exponencial) ===")
print(f"{'n':>12} | {'interpolation (ms)':>20} | {'binary (ms)':>13}")
print("-" * 52)

for n in SIZES:
    skewed = sorted(int(random.expovariate(1 / (n // 10))) for _ in range(n))
    # Pick existing element to ensure hit
    target = skewed[n // 2]

    t_interp = timeit.timeit(
        lambda d=skewed, t=target: interpolation_search(d, t), number=RUNS
    ) / RUNS * 1000

    t_binary = timeit.timeit(
        lambda d=skewed, t=target: binary_search_for_bench(d, t), number=RUNS
    ) / RUNS * 1000

    print(f"{n:>12,} | {t_interp:>20.4f} | {t_binary:>13.4f}")

## Exercicios

**Desafio 1:** Implemente `robust_interpolation_search(arr, target)` que usa interpolation search como estrategia principal, mas cai automaticamente para binary search quando detecta que a estimativa esta errando sistematicamente (ex: mais de 3 iteracoes sem reducao significativa do espaco de busca). Valide que o pior caso permanece O(log n).

**Desafio 2:** Implemente uma funcao que gera dois datasets de n=50.000 elementos (um uniforme, um distorcido com lei de Zipf), executa ambos os algoritmos e plota o numero de iteracoes por posicao do alvo (varredura de 0 a n-1) para mostrar visualmente onde cada algoritmo e superior.

In [ ]:
import math


# Desafio 1: interpolation search com fallback para binary
def robust_interpolation_search(arr, target, max_slow_iters=3):
    """
    Interpolation search with automatic fallback to binary search.
    Falls back when the search space is not shrinking fast enough,
    which indicates skewed data where interpolation degrades.

    Args:
        arr: Sorted list of numeric elements.
        target: Numeric value to search for.
        max_slow_iters: Max consecutive iterations with slow convergence.

    Returns:
        (index_or_minus1, mode_used, iterations)
    """
    low, high = 0, len(arr) - 1
    iterations = 0
    slow_count = 0
    prev_range = high - low + 1

    while low <= high and arr[low] <= target <= arr[high]:
        iterations += 1

        current_range = high - low + 1
        # Check convergence: if range reduced by less than log2 factor, it's slow
        if prev_range > 1 and current_range > prev_range / 2:
            slow_count += 1
        else:
            slow_count = 0
        prev_range = current_range

        if slow_count >= max_slow_iters:
            # Fallback: binary search for the remaining range
            sub = arr[low:high + 1]
            bl, bh = 0, len(sub) - 1
            while bl <= bh:
                iterations += 1
                mid = bl + (bh - bl) // 2
                if sub[mid] == target:
                    return low + mid, "binary_fallback", iterations
                elif sub[mid] < target:
                    bl = mid + 1
                else:
                    bh = mid - 1
            return -1, "binary_fallback", iterations

        if arr[high] == arr[low]:
            return (low if arr[low] == target else -1), "interpolation", iterations

        pos = low + int((target - arr[low]) * (high - low) / (arr[high] - arr[low]))

        if arr[pos] == target:
            return pos, "interpolation", iterations
        elif arr[pos] < target:
            low = pos + 1
        else:
            high = pos - 1

    return -1, "interpolation", iterations


# Test on uniform and skewed data
uniform = list(range(0, 10000, 2))
skewed = sorted([int(x ** 3) for x in range(1, 215)])  # cubic growth

target_u = uniform[len(uniform) * 3 // 4]
target_s = skewed[len(skewed) * 3 // 4]

idx_u, mode_u, iters_u = robust_interpolation_search(uniform, target_u)
idx_s, mode_s, iters_s = robust_interpolation_search(skewed, target_s)

print(f"Uniform  n={len(uniform)}: index={idx_u}, mode={mode_u}, iterations={iters_u}")
print(f"Skewed   n={len(skewed)}: index={idx_s}, mode={mode_s}, iterations={iters_s}")

## Referencias

1. Perl, Y., Itai, A., & Avni, H. "Interpolation search -- a log log n search". Communications of the ACM, 21(7), 550-553, 1978.
2. Knuth, D. E. "The Art of Computer Programming, Vol. 3: Sorting and Searching" (2nd ed.). Addison-Wesley, 1998, pp. 408-412.
3. Sedgewick, R., & Wayne, K. "Algorithms" (4th ed.). Addison-Wesley, 2011, pp. 382-383.